# PINN simple pour l'équation de Burgers 1D

On approxime $u(x,t)$ solution de

$$\partial_t u + u\,\partial_x u = \nu\,\partial_{xx} u, \qquad x \in [-1,1],\; t \in [0,1]$$

$$u(x,0) = -\sin(\pi x), \qquad u(-1,t) = u(1,t) = 0$$

(benchmark classique, $\nu = 0.01/\pi$) avec un réseau de neurones dont la loss combine :
- l'erreur sur la condition initiale et les bords (data loss)
- le résidu de l'EDP évalué par différentiation automatique (physics loss)

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

nu = 0.01 / np.pi

## Réseau : simple MLP (x, t) -> u

In [ ]:
class PINN(nn.Module):
    def __init__(self, n_hidden=20, n_layers=4):
        super().__init__()
        layers = [nn.Linear(2, n_hidden), nn.Tanh()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(n_hidden, n_hidden), nn.Tanh()]
        layers += [nn.Linear(n_hidden, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))


model = PINN()

## Points d'entraînement

- points de collocation dans le domaine (pour le résidu de l'EDP)
- points sur la condition initiale et les bords (pour la data loss)

In [ ]:
n_collocation = 2000
n_boundary = 100

x_f = torch.tensor(np.random.uniform(-1, 1, (n_collocation, 1)), dtype=torch.float32, requires_grad=True)
t_f = torch.tensor(np.random.uniform(0, 1, (n_collocation, 1)), dtype=torch.float32, requires_grad=True)

x_ic = torch.tensor(np.random.uniform(-1, 1, (n_boundary, 1)), dtype=torch.float32)
t_ic = torch.zeros_like(x_ic)
u_ic = -torch.sin(np.pi * x_ic)

t_bc = torch.tensor(np.random.uniform(0, 1, (n_boundary, 1)), dtype=torch.float32)
x_bc_left = -torch.ones_like(t_bc)
x_bc_right = torch.ones_like(t_bc)

## Résidu de l'EDP (autograd)

In [ ]:
def pde_residual(model, x, t, nu):
    u = model(x, t)
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t + u * u_x - nu * u_xx

## Entraînement

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
n_epochs = 3000

history = []
for epoch in range(n_epochs):
    optimizer.zero_grad()

    residual = pde_residual(model, x_f, t_f, nu)
    loss_f = torch.mean(residual**2)

    loss_ic = torch.mean((model(x_ic, t_ic) - u_ic) ** 2)
    loss_bc = torch.mean(model(x_bc_left, t_bc) ** 2) + torch.mean(model(x_bc_right, t_bc) ** 2)

    loss = loss_f + loss_ic + loss_bc
    loss.backward()
    optimizer.step()

    history.append(loss.item())
    if epoch % 500 == 0:
        print(f"epoch {epoch:4d}  loss={loss.item():.5f}  (pde={loss_f.item():.5f}, ic={loss_ic.item():.5f}, bc={loss_bc.item():.5f})")

## Visualisation

In [ ]:
plt.figure(figsize=(6, 4))
plt.semilogy(history)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Convergence du PINN")
plt.tight_layout()
plt.show()

In [ ]:
x_plot = np.linspace(-1, 1, 200)
times = [0.0, 0.25, 0.5, 0.75]

plt.figure(figsize=(8, 5))
for tv in times:
    x_t = torch.tensor(x_plot.reshape(-1, 1), dtype=torch.float32)
    t_t = torch.full_like(x_t, tv)
    with torch.no_grad():
        u_pred = model(x_t, t_t).numpy().flatten()
    plt.plot(x_plot, u_pred, label=f"t={tv}")

plt.xlabel("x")
plt.ylabel("u(x,t)")
plt.title("Solution prédite par le PINN à différents instants")
plt.legend()
plt.tight_layout()
plt.show()

On retrouve bien le comportement attendu : la condition initiale sinusoïdale se raidit progressivement vers un choc proche de $x=0$, formé par la compétition entre advection non-linéaire et diffusion visqueuse.